# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

# Data Loading

In [2]:
# Primary dataset: NT Government Region Populations 1986–2025
pop = pd.read_excel('nt-government-regions_1986-to-2025.xlsx')

# Secondary datasets: NT Crime Statistics
crime_nov = pd.read_csv('nt_crime_statistics_nov_2023.csv')   # 2008–2023
crime_dec = pd.read_csv('nt_crime_statistics_dec_2025.csv')   # 2023–2025

# Standardise column names to snake_case
for df in [crime_nov, crime_dec, pop]:
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# Preprocessing and Data Integration

### Split crime dataset by time period

In [3]:
# Nov 2023 file: 2008–2023; Dec 2025 file: 2023–2025
# Use Nov for 2008–2022, Dec for 2023–2025
crime_nov_hist = (crime_nov[crime_nov['year'] <= 2022]
                  .copy()
                  .rename(columns={'reporting_region': 'region'}))
crime_dec_new  = (crime_dec[crime_dec['reporting_region'] != 'Unknown']
                  .copy()
                  .rename(columns={'reporting_region': 'region',
                                   'offence_type_': 'offence_type'}))

### Offence category harmonisation

In [4]:
# Dec 2025 uses ABS numeric codes; map to Nov 2023 plain-English labels
CATEGORY_MAP = {
    '01 Homicide'                         : 'Homicide and related Offences',
    '02 Assault'                          : 'Acts intended to cause Injury',
    '03 Sexual offences'                  : 'Sexual assault and related offences',
    '04 Harm or endanger persons'         : 'Abduction - harassment and other offences against the person',
    '05 Robbery, blackmail, and extortion': 'Robbery - extortion and related offences',
    '061 Burglary - dwelling'             : 'House break-ins',
    '062 Burglary - non-residential'      : 'Commercial break-ins',
    '07 Theft'                            : 'Theft and related offences (other than MV)',
    '11 Property damage offences'         : 'Property Damage Offences',
}
crime_dec_new['offence_category'] = (
    crime_dec_new['offence_category']
    .map(CATEGORY_MAP)
    .fillna(crime_dec_new['offence_category'])
)
print("✓ Offence categories harmonised")

✓ Offence categories harmonised


### Region harmonisation

In [5]:
# Crime data uses 7 reporting regions; population data uses 6 NTG service regions.
# Mapping based on geographic alignment (see Assessment 1, p.3).
# This is what we termed "region harmonisation" in our project plan.
region_map = {
    'Alice Springs':  'Central Australia',
    'Darwin':         'Greater Darwin',
    'Palmerston':     'Greater Darwin',   # Palmerston is part of Greater Darwin
    'Katherine':      'Big Rivers',
    'Tennant Creek':  'Barkly',
    'Nhulunbuy':      'East Arnhem',
    'NT Balance':     'Top End'
}
for df in [crime_nov_hist, crime_dec_new]:
    df['pop_region'] = df['region'].map(region_map)

### Aggregate crime to region-year-month level

In [6]:
def aggregate_crime(df):
    """Aggregate offences by year, month, region, and offence category."""
    return (df.groupby(['year', 'month_number', 'pop_region', 'region', 'offence_category'])
             .agg(total_offences=('number_of_offences', 'sum'))
             .reset_index())

crime_all = (pd.concat([aggregate_crime(crime_nov_hist),
                        aggregate_crime(crime_dec_new)],
                       ignore_index=True)
               .drop_duplicates())

In [7]:
crime_rym = (crime_all
             .groupby(['year', 'month_number', 'pop_region', 'region'])
             .agg(total_offences=('total_offences', 'sum'))
             .reset_index())

### Alcohol and DV involvement flags

In [8]:
# Note: alcohol/DV flags are binary Yes/No — count rows flagged as 'Yes'
def get_involvement_flags(df):
    """Extract alcohol and domestic violence involvement counts."""
    df2 = df.copy()
    df2['alc'] = (df2['alcohol_involvement'].str.lower()
                  .str.contains('yes').fillna(False).astype(int))
    df2['dv']  = (df2['dv_involvement'].str.lower()
                  .str.contains('yes').fillna(False).astype(int))
    return (df2.groupby(['year', 'month_number', 'pop_region', 'region'])
               .agg(alc_offences=('alc', 'sum'),
                    dv_offences=('dv', 'sum'))
               .reset_index())

flags_all = (pd.concat([get_involvement_flags(crime_nov_hist),
                        get_involvement_flags(crime_dec_new)])
               .drop_duplicates())

crime_rym = crime_rym.merge(flags_all,
                             on=['year', 'month_number', 'pop_region', 'region'],
                             how='left')

### Aggregate population data

In [9]:
pop_total = (pop.groupby(['year', 'region'])['population']
               .sum()
               .reset_index()
               .rename(columns={'region': 'pop_region'}))

### Merge crime and population datasets

In [10]:
merged = (crime_rym
          .merge(pop_total, on=['year', 'pop_region'], how='left')
          .dropna(subset=['population']))

### Feature engineering

In [11]:
# Offence rate normalised per 1,000 residents (standard criminology metric)
merged['offence_rate_per_1000'] = (merged['total_offences'] / merged['population']) * 1000
# Proportional involvement rates (0–1)
merged['alc_rate'] = merged['alc_offences'] / merged['total_offences'].replace(0, np.nan)
merged['dv_rate']  = merged['dv_offences']  / merged['total_offences'].replace(0, np.nan)
# Log-transform population to reduce skewness (justified: population range is ~5k–250k)
merged['log_population'] = np.log(merged['population'])
# Year index for temporal trend modelling
merged['year_index'] = merged['year'] - merged['year'].min()
merged = merged.fillna(0)

### Aggregate crime to region-year-month-category level

In [12]:
pop_demo = (pop.groupby(['year', 'region'])
               .apply(lambda g: pd.Series({
                   'total_pop': g['population'].sum(),
                   'aboriginal_pop': g.loc[
                       g['aboriginal_status'].str.contains('Aboriginal', na=False),
                       'population'].sum()
               }))
               .reset_index()
               .rename(columns={'region': 'pop_region'}))
pop_demo['aboriginal_pct'] = (pop_demo['aboriginal_pop'] /
                               pop_demo['total_pop'] * 100)

merged2 = (merged
           .merge(pop_demo[['year', 'pop_region', 'aboriginal_pct']],
                  on=['year', 'pop_region'], how='left')
           .fillna(0))

print(f"Final merged dataset: {merged.shape[0]} rows × {merged.shape[1]} cols")
print(f"Year range: {merged['year'].min()}–{merged['year'].max()}")
print(f"Regions: {sorted(merged['region'].unique())}")
print(f"Null values:\n{merged.isnull().sum()}")

Final merged dataset: 1435 rows × 13 cols
Year range: 2008–2025
Regions: ['Alice Springs', 'Darwin', 'Katherine', 'NT Balance', 'Nhulunbuy', 'Palmerston', 'Tennant Creek']
Null values:
year                     0
month_number             0
pop_region               0
region                   0
total_offences           0
alc_offences             0
dv_offences              0
population               0
offence_rate_per_1000    0
alc_rate                 0
dv_rate                  0
log_population           0
year_index               0
dtype: int64


# Exploratory Data Analysis (RQ2)

In [13]:
# Prepare RQ2 yearly dataset
rq2 = (merged2.groupby(['year', 'pop_region', 'region'])
       .agg(total_offences    = ('total_offences',    'sum'),
            total_population  = ('population',        'first'),
            aboriginal_pct    = ('aboriginal_pct',    'first'),
            alc_rate          = ('alc_rate',           'mean'),
            dv_rate           = ('dv_rate',            'mean'))
       .reset_index())

rq2['offence_rate'] = rq2['total_offences'] / rq2['total_population'] * 1000
rq2['log_pop']      = np.log(rq2['total_population'])
rq2 = rq2.dropna()

print(f"RQ2 dataset: {rq2.shape[0]} rows × {rq2.shape[1]} cols")
print(f"Year range:  {rq2['year'].min()}–{rq2['year'].max()}")
print(f"Regions:     {sorted(rq2['region'].unique())}")
print()
print(rq2[['offence_rate','log_pop','aboriginal_pct','alc_rate','dv_rate','year']]
      .describe().round(3).to_string())

RQ2 dataset: 126 rows × 10 cols
Year range:  2008–2025
Regions:     ['Alice Springs', 'Darwin', 'Katherine', 'NT Balance', 'Nhulunbuy', 'Palmerston', 'Tennant Creek']

       offence_rate  log_pop  aboriginal_pct  alc_rate  dv_rate      year
count       126.000  126.000           126.0   126.000  126.000   126.000
mean        122.286   10.371           100.0     0.025    0.025  2016.500
std         114.773    1.056             0.0     0.025    0.025     5.209
min           1.866    8.873           100.0     0.002    0.002  2008.000
25%          26.294    9.660           100.0     0.006    0.006  2012.000
50%          81.405    9.970           100.0     0.015    0.015  2016.500
75%         166.113   11.753           100.0     0.041    0.043  2021.000
max         485.112   11.971           100.0     0.140    0.133  2025.000


In [14]:
# EDA Plot 1: Pairwise scatter — each predictor vs offence rate 
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA: Predictor Relationships with Offence Rate (RQ2)',
             fontsize=14, fontweight='bold')

COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2']
plot_vars = [
    ('log_pop',       'Log(Population)'),
    ('aboriginal_pct','Aboriginal Population %'),
    ('alc_rate',      'Alcohol Involvement Rate'),
    ('dv_rate',       'DV Involvement Rate'),
    ('year',          'Year'),
]

for idx, (var, label) in enumerate(plot_vars):
    ax = axes[idx // 3][idx % 3]
    for i, reg in enumerate(sorted(rq2['region'].unique())):
        sub = rq2[rq2['region'] == reg]
        ax.scatter(sub[var], sub['offence_rate'],
                   label=reg, alpha=0.7, s=35,
                   color=COLORS[i % len(COLORS)])
    # Regression line
    if rq2[var].std() > 0:
        m, b, r, p, _ = stats.linregress(rq2[var], rq2['offence_rate'])
        xline = np.linspace(rq2[var].min(), rq2[var].max(), 100)
        ax.plot(xline, m * xline + b, 'k--', lw=1.5,
                label=f'r={r:+.2f} (p={p:.3f})')
    else:
        ax.set_title(ax.get_title() + ' [constant — skip]', fontsize=9)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Offence Rate per 1,000', fontsize=10)
    ax.set_title(f'{label} vs Offence Rate', fontsize=10, fontweight='bold')
    ax.legend(fontsize=6, ncol=2)

# Hide the unused 6th subplot
axes[1][2].axis('off')

plt.tight_layout()
plt.savefig('rq2_eda_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ rq2_eda_scatter.png saved")

✓ rq2_eda_scatter.png saved


In [15]:
# EDA Plot 2: Correlation heatmap 
fig, ax = plt.subplots(figsize=(8, 6))
corr_cols = ['offence_rate', 'log_pop', 'aboriginal_pct',
             'alc_rate', 'dv_rate', 'year']
corr_df = rq2[corr_cols].corr()
sns.heatmap(corr_df, ax=ax, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5,
            annot_kws={'size': 11})
ax.set_title('Feature Correlation Matrix — RQ2 Predictors',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rq2_eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ rq2_eda_correlation.png saved")

✓ rq2_eda_correlation.png saved


In [16]:
# EDA Plot 3: Offence rate by region — box plot
fig, ax = plt.subplots(figsize=(10, 5))
region_order = (rq2.groupby('region')['offence_rate']
                .median().sort_values(ascending=False).index.tolist())
data_bp = [rq2[rq2['region'] == r]['offence_rate'].values
           for r in region_order]
bp = ax.boxplot(data_bp, labels=region_order, patch_artist=True,
                medianprops={'color': 'black', 'lw': 2})
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(COLORS[i % len(COLORS)])
    patch.set_alpha(0.75)
ax.set_ylabel('Offence Rate per 1,000 Population', fontsize=11)
ax.set_title('Offence Rate Distribution by Region (sorted by median)',
             fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=30, labelsize=9)
plt.tight_layout()
plt.savefig('rq2_eda_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ rq2_eda_boxplot.png saved")

✓ rq2_eda_boxplot.png saved


# RQ 2 - Multiple Linear Regression

In [17]:
# RQ2: To what extent do demographic indicators explain variation in recorded offence rates across regions?

rq2 = (merged2.groupby(['year', 'pop_region', 'region'])
       .agg(total_offences=('total_offences', 'sum'),
            total_population=('population', 'first'),
            aboriginal_pct=('aboriginal_pct', 'first'),
            alc_rate=('alc_rate', 'mean'),
            dv_rate=('dv_rate', 'mean'))
       .reset_index())
rq2['offence_rate'] = rq2['total_offences'] / rq2['total_population'] * 1000
rq2['log_pop']      = np.log(rq2['total_population'])
rq2 = rq2.dropna()

features_rq2 = ['log_pop', 'aboriginal_pct', 'alc_rate', 'dv_rate', 'year']
X2 = rq2[features_rq2].values
y2 = rq2['offence_rate'].values

scaler2 = StandardScaler()
X2s = scaler2.fit_transform(X2)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2s, y2, test_size=0.2, random_state=42)

mlr = LinearRegression()
mlr.fit(X2_train, y2_train)
y2_pred = mlr.predict(X2_test)

mae2  = mean_absolute_error(y2_test, y2_pred)
rmse2 = np.sqrt(mean_squared_error(y2_test, y2_pred))
r2_2  = r2_score(y2_test, y2_pred)

cv_r2  = cross_val_score(LinearRegression(), X2s, y2, cv=5, scoring='r2')
cv_mae = cross_val_score(LinearRegression(), X2s, y2, cv=5,
                          scoring='neg_mean_absolute_error')

# Statistical test: t-test on residuals (H0: mean residual = 0)
residuals2 = y2_test - y2_pred
t_stat2, p_val2 = stats.ttest_1samp(residuals2, 0)

print("\n=== RQ2: Multiple Linear Regression Results ===")
print(f"  MAE  = {mae2:.3f} offences/1,000")
print(f"  RMSE = {rmse2:.3f} offences/1,000")
print(f"  R²   = {r2_2:.3f}")
print(f"  5-Fold CV R²  = {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"  5-Fold CV MAE = {-cv_mae.mean():.3f} ± {cv_mae.std():.3f}")
print(f"  T-test residuals: t={t_stat2:.4f}, p={p_val2:.4f}")
print(f"  Coefficients:")
for feat, coef in zip(features_rq2, mlr.coef_):
    print(f"    {feat:20s}: {coef:+.4f}")


=== RQ2: Multiple Linear Regression Results ===
  MAE  = 56.300 offences/1,000
  RMSE = 80.463 offences/1,000
  R²   = 0.643
  5-Fold CV R²  = 0.539 ± 0.175
  5-Fold CV MAE = 59.145 ± 13.823
  T-test residuals: t=0.9850, p=0.3341
  Coefficients:
    log_pop             : -42.7567
    aboriginal_pct      : -0.0000
    alc_rate            : -205.8586
    dv_rate             : +201.9530
    year                : +2.3397


# Assumption Checks

In [18]:
# Assumption 1: Linearity (Pearson r, each predictor vs target) 
print("=== ASSUMPTION 1: LINEARITY ===")
print("Pearson r between each predictor and offence_rate:")
for f in features_rq2:
    if rq2[f].std() > 0:
        r, p = stats.pearsonr(rq2[f], rq2['offence_rate'])
        flag = '✓' if abs(r) > 0.1 else '⚠ Weak'
        print(f"  {f:20s}: r={r:+.3f}, p={'<0.001' if p<0.001 else f'{p:.4f}'}  {flag}")
    else:
        print(f"  {f:20s}: constant — skip")

=== ASSUMPTION 1: LINEARITY ===
Pearson r between each predictor and offence_rate:
  log_pop             : r=-0.364, p=<0.001  ✓
  aboriginal_pct      : constant — skip
  alc_rate            : r=+0.094, p=0.2944  ⚠ Weak
  dv_rate             : r=+0.323, p=<0.001  ✓
  year                : r=+0.028, p=0.7577  ⚠ Weak


In [19]:
# Assumption 2: Normality of residuals 
print("=== ASSUMPTION 2: NORMALITY OF RESIDUALS ===")
sw_stat, sw_p  = stats.shapiro(residuals2)
jb_stat, jb_p  = stats.jarque_bera(residuals2)
print(f"  Shapiro-Wilk:  W={sw_stat:.4f},  p={sw_p:.4f}  {'PASS' if sw_p>0.05 else 'BORDERLINE'}")
print(f"  Jarque-Bera:   stat={jb_stat:.4f}, p={jb_p:.4f}  {'PASS' if jb_p>0.05 else 'BORDERLINE'}")
print(f"  Skewness:      {stats.skew(residuals2):.3f}")
print(f"  Kurtosis:      {stats.kurtosis(residuals2):.3f}")
print(f"  Mean residual: {residuals2.mean():.6f}  (should be ~0)")
print()
print("  Note: With N=126, the Central Limit Theorem partially mitigates")
print("  mild non-normality — coefficient estimates remain asymptotically valid.")

=== ASSUMPTION 2: NORMALITY OF RESIDUALS ===
  Shapiro-Wilk:  W=0.8836,  p=0.0069  BORDERLINE
  Jarque-Bera:   stat=9.2473, p=0.0098  BORDERLINE
  Skewness:      1.248
  Kurtosis:      1.519
  Mean residual: 15.552710  (should be ~0)

  Note: With N=126, the Central Limit Theorem partially mitigates
  mild non-normality — coefficient estimates remain asymptotically valid.


In [20]:
# Assumption 3: Homoscedasticity
print("=== ASSUMPTION 3: HOMOSCEDASTICITY ===")
r_bp, p_bp = stats.pearsonr(np.abs(residuals2), y2_pred)
print(f"  |residuals| vs predicted values: r={r_bp:.4f}, p={p_bp:.4f}")
print(f"  {'PASS' if p_bp>0.05 else 'PARTIAL — some variance growth at higher predicted values'}")
print()
print("  Limitation: Heteroscedasticity is common in regional crime data")
print("  where small remote communities behave very differently from large cities.")

=== ASSUMPTION 3: HOMOSCEDASTICITY ===
  |residuals| vs predicted values: r=0.5630, p=0.0027
  PARTIAL — some variance growth at higher predicted values

  Limitation: Heteroscedasticity is common in regional crime data
  where small remote communities behave very differently from large cities.


In [21]:
# Assumption 4: No multicollinearity (VIF) 
print("=== ASSUMPTION 4: MULTICOLLINEARITY (VIF) ===")
for i, f in enumerate(features_rq2):
    col = X2s[:, i]
    if col.std() < 1e-10:
        print(f"  {f:20s}: constant — skip")
        continue
    mask = [j for j in range(len(features_rq2)) if j != i and X2s[:, j].std() > 1e-10]
    r2_vif = r2_score(col,
                      LinearRegression().fit(X2s[:, mask], col).predict(X2s[:, mask]))
    vif = 1 / (1 - r2_vif) if r2_vif < 0.9999 else 999
    flag = '✓ OK' if vif < 5 else ('⚠ Moderate' if vif < 10 else '✗ High')
    print(f"  {f:20s}: VIF={vif:.2f}  {flag}")
print()
print("  Note: Moderate VIF for alc_rate and dv_rate reflects genuine")
print("  real-world correlation between alcohol and DV involvement.")
print("  Ridge regression was tested as an alternative (see Section 7).")

=== ASSUMPTION 4: MULTICOLLINEARITY (VIF) ===
  log_pop             : VIF=1.52  ✓ OK
  aboriginal_pct      : constant — skip
  alc_rate            : VIF=8.95  ⚠ Moderate
  dv_rate             : VIF=8.92  ⚠ Moderate
  year                : VIF=1.02  ✓ OK

  Note: Moderate VIF for alc_rate and dv_rate reflects genuine
  real-world correlation between alcohol and DV involvement.
  Ridge regression was tested as an alternative (see Section 7).


In [22]:
# Assumption 5: Independence of errors (Durbin-Watson) 
print("=== ASSUMPTION 5: INDEPENDENCE OF ERRORS ===")
dw = np.sum(np.diff(residuals2)**2) / np.sum(residuals2**2)
print(f"  Durbin-Watson = {dw:.4f}")
print(f"  (Ideal=2.0; acceptable range 1.5–2.5)")
flag = '✓ PASS' if 1.5 < dw < 2.5 else '⚠ Check autocorrelation'
print(f"  {flag}")

=== ASSUMPTION 5: INDEPENDENCE OF ERRORS ===
  Durbin-Watson = 2.2250
  (Ideal=2.0; acceptable range 1.5–2.5)
  ✓ PASS


In [23]:
# Assumption 6: Sample size 
print("=== ASSUMPTION 6: ADEQUATE SAMPLE SIZE ===")
n_obs, n_feat = len(rq2), len(features_rq2)
ratio = n_obs / n_feat
print(f"  N = {n_obs}, k = {n_feat}, ratio = {ratio:.1f} obs per feature")
print(f"  Rule of thumb: need ≥ 10 obs per feature")
print(f"  {'✓ PASS' if ratio >= 10 else '✗ FAIL'}  ({ratio:.1f} ≥ 10)")

=== ASSUMPTION 6: ADEQUATE SAMPLE SIZE ===
  N = 126, k = 5, ratio = 25.2 obs per feature
  Rule of thumb: need ≥ 10 obs per feature
  ✓ PASS  (25.2 ≥ 10)


# Visualisation

In [24]:
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle('RQ2: Multiple Linear Regression — Full Diagnostic Report',
             fontsize=15, fontweight='bold')

# ── Plot 1: Actual vs Predicted ──────────────────────────────────────
ax = axes[0, 0]
ax.scatter(y2_test, y2_pred, alpha=0.75, color='#1f77b4',
           s=60, edgecolors='white', lw=0.5)
lo = min(y2_test.min(), y2_pred.min())
hi = max(y2_test.max(), y2_pred.max())
ax.plot([lo, hi], [lo, hi], 'r--', lw=2, label='Perfect Fit (y=x)')
ax.set_xlabel('Actual Offence Rate per 1,000', fontsize=11)
ax.set_ylabel('Predicted Offence Rate per 1,000', fontsize=11)
ax.set_title(f'Actual vs Predicted\n'
             f'R² = {r2_score(y2_test,y2_pred):.3f}  |  '
             f'MAE = {mean_absolute_error(y2_test,y2_pred):.1f}  |  '
             f'RMSE = {np.sqrt(mean_squared_error(y2_test,y2_pred)):.1f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

# ── Plot 2: Residual plot ────────────────────────────────────────────
ax = axes[0, 1]
ax.scatter(y2_pred, residuals2, alpha=0.7, color='#9467bd',
           s=55, edgecolors='white', lw=0.5)
ax.axhline(0, color='red', lw=2, linestyle='--', label='Zero residual')
ax.set_xlabel('Predicted Offence Rate', fontsize=11)
ax.set_ylabel('Residuals (Actual − Predicted)', fontsize=11)
ax.set_title('Residual Plot\n(random scatter around 0 = homoscedasticity satisfied)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

# ── Plot 3: Standardised feature coefficients ────────────────────────
ax = axes[1, 0]
feat_labels = ['Log(Population)', 'Aboriginal %', 'Alcohol Rate',
               'DV Rate', 'Year']
coefs  = mlr.coef_
bar_c  = ['#1f77b4' if c >= 0 else '#d62728' for c in coefs]
bars   = ax.barh(feat_labels, coefs, color=bar_c, edgecolor='black', alpha=0.85)
ax.axvline(0, color='black', lw=1.2)
for bar, val in zip(bars, coefs):
    ax.text(val + (1 if val >= 0 else -1),
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.2f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=10)
ax.set_xlabel('Standardised Coefficient', fontsize=11)
ax.set_title('Feature Coefficients\n'
             '(Blue = positive effect on crime rate  |  Red = negative effect)',
             fontsize=11, fontweight='bold')

# ── Plot 4: 5-Fold Cross-Validation R² ──────────────────────────────
ax = axes[1, 1]
fold_colors = ['#2ca02c' if v >= cv_r2.mean() else '#d62728' for v in cv_r2]
ax.bar(range(1, 6), cv_r2, color=fold_colors, alpha=0.85, edgecolor='black')
ax.axhline(cv_r2.mean(), color='navy', lw=2, linestyle='--',
           label=f'Mean R² = {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Fold', fontsize=11)
ax.set_ylabel('R²', fontsize=11)
ax.set_xticks(range(1, 6))
ax.set_title('5-Fold Cross-Validation R² Scores\n'
             '(Green = above mean  |  Red = below mean)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('rq2_regression_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ rq2_regression_diagnostics.png saved")

✓ rq2_regression_diagnostics.png saved
